In [0]:
import platform
import requests
import shlex
import zipfile
import tempfile
import shutil
import subprocess
import os
import hashlib
import yaml
from dbruntime.databricks_repl_context import get_context

cli_version = "0.295.0"
terraform_version = "1.5.5"
provider_version = "1.111.0"
uv_version = "0.6.10"
cwd = "/Repos/Beth_Development/Demo_DOT"
args = "bundle summary --show-full-config --include-locations"
tmp_file = "/Workspace/Users/beth.ramsey@trace3.com/.bundle/databricks_deployment/.output/load-full-config-output-r9qsga75tuo.json"
dev_cli_snippet = ""
artifact_download_error_prefix = "DABsRunnerError: Failed to download CLI dependency"
tmp_downloading_fallback_error = "DabsRunnerErrorFallback: Downloading error, using fallback URL"
allow_python_execution = "false"
enable_yaml_sync = "false"
enable_direct_engine = "false"
include_stderr_in_error = "false"
cli_upstream = "dabs-in-workspace"

def create_uv_wrapper(uv_exec_path, uv_version):
  uv_wrapper_code = f"""#!/bin/bash -e
if [[ ! -f "{uv_exec_path}" ]]; then
  curl -LsSf https://astral.sh/uv/{uv_version}/install.sh | sh
  exec "$HOME/.local/bin/uv" "$@"
fi
exec "{uv_exec_path}" "$@"
"""
  uv_wrapper_path = "uv"
  with open(uv_wrapper_path, "w") as file:
    file.write(uv_wrapper_code)
  os.chmod(uv_wrapper_path, 0o755)
  os.environ["PATH"] = os.environ["PATH"] + os.pathsep + os.getcwd()

ctx = get_context()
token = ctx.apiToken
workspace_url = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"
cpu_arch = "linux_arm64" if platform.machine() == "aarch64" else "linux_amd64"

def get_download_url(file_api_path, fallback_download_url):
  api_create_download_url = f"{workspace_url}/api/2.0/fs/create-download-url"
  headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
  data = {"path": file_api_path}
  response = requests.post(api_create_download_url, headers=headers, json=data)
  if response.status_code != 200:
    print(f"{tmp_downloading_fallback_error}: {file_api_path} HTTP {response.status_code} {response.text}")
    if fallback_download_url is None:
      raise RuntimeError(f"Failed to download {file_api_path}: HTTP {response.status_code}")
    return fallback_download_url
  return response.json()["url"]

def download_and_extract(file_url, dest_dir):
  response = requests.get(file_url)
  if response.status_code != 200:
    raise RuntimeError(f"HTTP {response.status_code} while downloading artifact: {response.text}")
  tmp_file = f"{dest_dir}.zip"

  with open(tmp_file, "wb") as zip_file:
      zip_file.write(response.content)
  with zipfile.ZipFile(tmp_file, 'r') as zip_ref:
      zip_ref.extractall(dest_dir)

def get_venv_path_from_databricks_yml(cwd):
  databricks_yml_path = f"{cwd}/databricks.yml"
  with open(databricks_yml_path, "r") as f:
    data = yaml.safe_load(f)
    return data.get("python", {}).get("venv_path", ".venv")

def run_cli_command(cli_command, cwd, tmp_file_path, include_stderr):
  stderr_arg = subprocess.PIPE if include_stderr else None
  if tmp_file_path:
    os.makedirs(os.path.dirname(tmp_file_path), exist_ok=True)
    result = subprocess.run(cli_command, cwd=cwd, stdout=subprocess.PIPE, stderr=stderr_arg, text=True)
    with open(tmp_file_path, 'w') as output_file:
        output_file.write(result.stdout)
        print(result.stdout)
  else:
    result = subprocess.run(cli_command, cwd=cwd, stderr=stderr_arg, text=True)
  if result.returncode != 0:
    error_msg = f"Command {result.args[:3]} failed with exit code {result.returncode}"
    if include_stderr and result.stderr:
      error_msg += f":\n{result.stderr}"
    raise RuntimeError(error_msg)

  # For Genie Code notebook, flush async I/O so bundle runner notebook gets correct state
  if cli_upstream == "genie-code":
    try:
      with open("/Workspace/.proc/self/status", "r") as status_file:
        status_file.read()
    except Exception as e:
      print(f"Warning: Failed to read /Workspace/.proc/self/status: {e}")

def download_artifact_if_not_exists(input_file_name, binary_name, dest_path, tmp_dir, fallback_download_url):
  if os.path.exists(dest_path):
    return

  try:
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    path_prefix = f"/DatabricksInternal/Jobs/ArtifactRepository/public/databricks-cli/{cli_version}"
    download_url = get_download_url(f"{path_prefix}/{input_file_name}.zip", fallback_download_url)

    tmp_archive_dir = f"{tmp_dir}/{input_file_name}"
    download_and_extract(download_url, tmp_archive_dir)

    src_path = f"{tmp_archive_dir}/{binary_name}"
    if not os.path.exists(src_path):
      available = ", ".join(os.listdir(tmp_archive_dir)) if os.path.isdir(tmp_archive_dir) else "(missing archive directory)"
      raise RuntimeError(f"Expected binary '{binary_name}' not found in archive contents: {available}")

    shutil.move(src_path, dest_path)
    os.chmod(dest_path, 0o755)
  except Exception as e:
    error_msg = f"{artifact_download_error_prefix} {input_file_name}/{binary_name} CLI v{cli_version}, TF v{terraform_version}, TF provider v{provider_version}: {str(e)}"
    raise RuntimeError(f"{error_msg}")

tmp_base = os.path.expanduser("~/tmp")
if not os.path.exists(tmp_base):
  os.makedirs(tmp_base)
os.chmod(tmp_base, 0o700)

with tempfile.TemporaryDirectory(dir=tmp_base) as tmpdirname:
    cli_exec_path = f"{tmp_base}/databricks_{cli_version}_{cpu_arch}"
    download_artifact_if_not_exists(
      f"databricks_cli_{cpu_arch}",
      "databricks",
      cli_exec_path,
      tmpdirname,
      f"https://github.com/databricks/cli/releases/download/v{cli_version}/databricks_cli_{cli_version}_{cpu_arch}.zip"
    )

    terraform_base = f"{tmp_base}/terraform"
    terraform_exec_path = f"{terraform_base}/terraform_{terraform_version}_{cpu_arch}"
    download_artifact_if_not_exists(
      f"terraform_{cpu_arch}",
      "terraform",
      terraform_exec_path,
      tmpdirname,
      f"https://releases.hashicorp.com/terraform/{terraform_version}/terraform_{terraform_version}_{cpu_arch}.zip"
    )

    provider_mirror_base = f"{terraform_base}/providers"
    provider_mirror_dir = f"{provider_mirror_base}/registry.terraform.io/databricks/databricks/{provider_version}/{cpu_arch}"
    os.makedirs(provider_mirror_dir, exist_ok=True)
    provider_exec_path = f"{provider_mirror_dir}/terraform-provider-databricks_v{provider_version}"
    download_artifact_if_not_exists(
      f"terraform_provider_{cpu_arch}",
      f"terraform-provider-databricks_v{provider_version}",
      provider_exec_path,
      tmpdirname,
      f"https://github.com/databricks/terraform-provider-databricks/releases/download/v{provider_version}/terraform-provider-databricks_{provider_version}_{cpu_arch}.zip"
    )

    uv_base = f"{tmp_base}/uv"
    uv_exec_path = f"{uv_base}/uv_{uv_version}_{cpu_arch}"
    try:
        download_artifact_if_not_exists(
          f"uv_{cpu_arch}",
          "uv",
          uv_exec_path,
          tmpdirname,
          None
        )
    except Exception:
        pass
    create_uv_wrapper(uv_exec_path, uv_version)

    terraform_cli_config_path = f"{terraform_base}/terraform-cli-config"
    if not os.path.exists(terraform_cli_config_path):
      terraform_cli_config = f'''provider_installation {{
  filesystem_mirror {{
    path    = "{provider_mirror_base}"
    include = ["registry.terraform.io/databricks/databricks"]
  }}
  direct {{
    exclude = ["registry.terraform.io/databricks/databricks"]
  }}
}}
disable_checkpoint = true
disable_checkpoint_signature = true
'''

      with open(terraform_cli_config_path, 'w') as f:
        f.write(terraform_cli_config)

    cwd = os.path.normpath(f"/Workspace{cwd}")

    os.environ["DATABRICKS_TF_VERSION"] = terraform_version
    os.environ["DATABRICKS_TF_EXEC_PATH"] = terraform_exec_path
    os.environ["DATABRICKS_TF_PROVIDER_VERSION"] = provider_version
    os.environ["DATABRICKS_TF_CLI_CONFIG_FILE"] = terraform_cli_config_path
    os.environ["TF_CLI_CONFIG_FILE"] = terraform_cli_config_path

    if dev_cli_snippet:
      cli_path = cli_exec_path
      exec(dev_cli_snippet, globals(), locals())

    os.environ["DATABRICKS_BUNDLE_TMP"] = f"{tmp_base}/bundle-2-{hashlib.sha256((cwd).encode()).hexdigest()}"
    os.environ["DATABRICKS_TOKEN"] = token
    os.environ["DATABRICKS_CLI_UPSTREAM"] = cli_upstream

    if enable_yaml_sync == "true":
      os.environ["DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC"] = "1"
    else:
      os.environ.pop("DATABRICKS_BUNDLE_ENABLE_EXPERIMENTAL_YAML_SYNC", None)

    if enable_direct_engine == "true":
      os.environ["DATABRICKS_BUNDLE_ENGINE"] = "direct"
    else:
      os.environ.pop("DATABRICKS_BUNDLE_ENGINE", None)

    if allow_python_execution == "true":
      venv_path = get_venv_path_from_databricks_yml(cwd)
      venv_folder_path = os.path.normpath(f"{cwd}/{venv_path}")
      venv_bin_path = f"{venv_folder_path}/bin"

      os.makedirs(venv_bin_path, exist_ok=True)

      python3_wrapper_path = f"{venv_bin_path}/python3"
      if not os.path.exists(python3_wrapper_path):
        with open(python3_wrapper_path, "w") as f:
          f.write('#!/usr/bin/env bash\nexec python3 "$@"\n')
        os.chmod(python3_wrapper_path, 0o755)

      pyvenv_cfg_path = f"{venv_folder_path}/pyvenv.cfg"
      if not os.path.exists(pyvenv_cfg_path):
        with open(pyvenv_cfg_path, "w") as f:
          pass

      pyproject_path = f"{cwd}/pyproject.toml"
      uv_pip_install_args = [uv_exec_path, "pip", "install", "--python", python3_wrapper_path, "--link-mode=copy"]

      override_file = None
      if os.path.exists(pyproject_path):
        uv_pip_install_args.extend(["-r", pyproject_path, "--group", "dev"])
      else:
        uv_pip_install_args.append(f"databricks-bundles=={cli_version}")

      try:
        subprocess.run(uv_pip_install_args, check=True, cwd=cwd, stdout=subprocess.DEVNULL)
      finally:
        if override_file and os.path.exists(override_file):
          os.remove(override_file)
      os.environ.pop("DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION", None)
    else:
      os.environ["DATABRICKS_BUNDLE_RESTRICTED_CODE_EXECUTION"] = "1"

    if "DATABRICKS_CLUSTER_ID" in os.environ:
      del os.environ["DATABRICKS_CLUSTER_ID"]

    with tempfile.NamedTemporaryFile(mode='w+', dir=tmpdirname, delete=True) as databrickscfg_file:
        profile_name = "workspace_profile"
        databrickscfg_file.write(f"[workspace_profile]\nhost = {workspace_url}")
        databrickscfg_file.flush()
        os.environ["DATABRICKS_CONFIG_FILE"] = databrickscfg_file.name

        cli_command = [cli_exec_path] + shlex.split(args) + ["-p", profile_name]
        tmp_file_path = tmp_file

        run_cli_command(cli_command, cwd, tmp_file_path, include_stderr_in_error == "true")
None
